# Estimación del valor de mercado de futbolistas profesionales

## Notebook 1 — Adquisición, auditoría y construcción del dataset analítico

**Trabajo Fin de Máster** · Máster en Big Data, Data Science e Inteligencia Artificial · Universidad Complutense de Madrid

---

### Qué hace este notebook

| | |
|---|---|
| **Consume** | Los 12 ficheros CSV del conjunto *Football Data from Transfermarkt* |
| **Produce** | `data/processed/dataset_analitico.parquet` |
| **Siguiente** | `02_modelos.ipynb`, que parte del parquet generado aquí |

### Estructura

1. **Configuración y adquisición** — descarga reproducible y congelación de una instantánea
2. **Comprensión de las tablas** — diccionario de datos y decisión sobre cada columna
3. **Auditoría de calidad** — integridad, consistencia y cobertura de los datos en crudo
4. **Construcción del dataset analítico** — de 1,9 M de apariciones a una fila por jugador y temporada
5. **Verificación de salida** — comprobaciones sobre el conjunto generado

### Nota metodológica

Este notebook **no contiene análisis exploratorio**. La exploración se realiza sobre el
dataset analítico ya construido y se recoge en el notebook 2, puesto que la unidad de
análisis del estudio es el jugador-temporada y esa unidad no existe en los datos en crudo.

Lo que sí se realiza aquí es una **auditoría de calidad**, que responde a una pregunta
distinta: si los datos de partida soportan la transformación que se pretende aplicar.

---
# 1. Configuración y adquisición

## 1.1. Configuración del entorno

Se fija la semilla de aleatoriedad, se definen las rutas del proyecto mediante `pathlib`
 y se registran las versiones
de las librerías empleadas, garantizando así su reproducibilidad.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

# --- Reproducibilidad -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Rutas del proyecto -----------------------------------------------------
PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW       = PROJ / "data" / "raw"
INTERIM   = PROJ / "data" / "interim"
PROCESSED = PROJ / "data" / "processed"
MODELS    = PROJ / "models"
FIGURES   = PROJ / "reports" / "figures"

for p in (RAW, INTERIM, PROCESSED, MODELS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

# --- Presentación -----------------------------------------------------------
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Proyecto:", PROJ)
print("Datos en:", RAW)

Proyecto: e:\PEPO\PEPO\MASTER\TFM\tfm_estimacionValorMercadoFutbolista\entrega\implementacion\src
Datos en: e:\PEPO\PEPO\MASTER\TFM\tfm_estimacionValorMercadoFutbolista\entrega\implementacion\src\data\raw


In [2]:
# Versiones del entorno (evidencia de reproducibilidad para el Anexo)
%load_ext watermark
%watermark -v -m -p pandas,numpy,pyarrow

Python implementation: CPython
Python version       : 3.11.15
IPython version      : 9.16.1

pandas : 3.0.5
numpy  : 2.4.6
pyarrow: 25.0.1

Compiler    : MSC v.1942 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : AMD64 Family 23 Model 96 Stepping 1, AuthenticAMD
CPU cores   : 16
Architecture: 64bit



## 1.2. Descarga y congelación de la instantánea

Este conjunto de datos se actualiza **semanalmente**, por lo que si el análisis siempre leyera
directamente de la última versión disponible, los resultados dejarían de ser reproducibles:
las valoraciones cambiarían y las cifras de la memoria no coincidirían con una reejecución
posterior.

Para evitarlo se descarga una única vez mediante `kagglehub` y se **congela una instantánea
fechada** en `data/raw/`.

In [3]:
import shutil
from datetime import date
import kagglehub

DATASET  = "davidcariboo/player-scores"
SNAPSHOT = RAW / f"snapshot_{date.today():%Y%m%d}"

existentes = sorted(RAW.glob("snapshot_*"))

if existentes:
    DATA = existentes[-1]
    print(f"Ya existe una instantánea: {DATA.name}. No se descarga de nuevo.")
else:
    print("Descargando desde Kaggle...")
    origen = Path(kagglehub.dataset_download(DATASET))

    SNAPSHOT.mkdir(parents=True)
    n = 0
    for csv in origen.rglob("*.csv"):
        shutil.copy2(csv, SNAPSHOT / csv.name)
        n += 1

    (SNAPSHOT / "_METADATA.txt").write_text(
        f"dataset: {DATASET}\n"
        f"fecha_descarga: {date.today():%Y-%m-%d}\n"
        f"origen_cache: {origen}\n"
        f"ficheros_csv: {n}\n",
        encoding="utf-8",
    )
    DATA = SNAPSHOT
    print(f"Copiados {n} ficheros CSV a {DATA}")

print("\n--- Instantánea de trabajo ---")
print((DATA / "_METADATA.txt").read_text(encoding="utf-8"))

Descargando desde Kaggle...


100%|██████████| 234M/234M [00:06<00:00, 37.9MB/s] 

Extracting files...


Copiados 12 ficheros CSV a e:\PEPO\PEPO\MASTER\TFM\tfm_estimacionValorMercadoFutbolista\entrega\implementacion\src\data\raw\snapshot_20260909

--- Instantánea de trabajo ---
dataset: davidcariboo/player-scores
fecha_descarga: 2026-09-09
origen_cache: C:\Users\Asus\.cache\kagglehub\datasets\davidcariboo\player-scores\versions\679
ficheros_csv: 12



## 1.3. Inventario de ficheros

Volumetría del conjunto de partida. El recuento de filas se realiza sin cargar los ficheros
en memoria, leyéndolos línea a línea, para solo cargar aquellos con los que realmente vamos a trabajar.

In [4]:
def contar_filas(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        return sum(1 for _ in f) - 1

inventario = pd.DataFrame([
    {"tabla": f.stem,
     "filas": contar_filas(f)}
    for f in sorted(DATA.glob("*.csv"))
]).sort_values("filas", ascending=False, ignore_index=True)

print(f"Total: {inventario['filas'].sum():,} filas")
inventario

Total: 7,497,433 filas


,tabla,filas
0,game_lineups,3179016
1,appearances,1894350
2,game_events,1274469
3,player_valuations,656301
4,club_games,177916
5,transfers,175165
6,games,88958
7,players,50149
8,clubs,796
9,countries,124


---
# 2. Comprensión de las tablas

## 2.1. Diccionario de tablas

El conjunto está organizado como un **modelo relacional de 12 tablas** con distintos niveles
de granularidad. Comprender qué representa cada fila en cada tabla es imprescindible antes de
cualquier transformación, ya que determina si una tabla debe **agregarse** o simplemente
**unirse**.

| Tabla | Una fila representa | Uso en este trabajo |
|---|---|---|
| `competitions` | Una competición | Filtro de ligas y tipo de torneo |
| `games` | Un partido | Aporta la temporada de cada partido |
| `clubs` | Un club | Contexto (aforo) |
| `players` | Un jugador | Perfil |
| `player_valuations` | Una revisión del valor de un jugador | **Variable objetivo** |
| `appearances` | Un jugador en un partido | Núcleo del rendimiento |
| `game_events` | Un evento dentro de un partido | Fuera del alcance |
| `game_lineups` | Un jugador en la alineación de un partido | Titularidades y capitanía |
| `club_games` | Un club en un partido | Forma del club por temporada |
| `transfers` | Un traspaso | Fuera del alcance |
| `countries` | Un país | Fuera del alcance |
| `player_teammates_played_with` | Coincidencias entre jugadores | Fuera del alcance |

**Anotación:** `appearances` debe agregarse (un jugador con 34 partidos ocupa
34 filas y debe quedar en una), mientras que `players` o `clubs` se unen como tablas
de referencia, ya que tienen una fila por entidad.

## 2.2. Esquemas reales de las tablas

In [5]:
for f in sorted(DATA.glob("*.csv")):
    cols = pd.read_csv(f, nrows=100, encoding="utf-8", low_memory=False).columns
    print(f"\n=== {f.stem} ({len(cols)} columnas) ===")
    print(", ".join(cols))


=== appearances (13 columnas) ===
appearance_id, game_id, player_id, player_club_id, player_current_club_id, date, player_name, competition_id, yellow_cards, red_cards, goals, assists, minutes_played

=== club_games (11 columnas) ===
game_id, club_id, own_goals, own_position, own_manager_name, opponent_id, opponent_goals, opponent_position, opponent_manager_name, hosting, is_win

=== clubs (17 columnas) ===
club_id, club_code, name, domestic_competition_id, total_market_value, squad_size, average_age, foreigners_number, foreigners_percentage, national_team_players, stadium_name, stadium_seats, net_transfer_record, coach_name, last_season, filename, url

=== competitions (11 columnas) ===
competition_id, competition_code, name, sub_type, type, country_id, country_name, domestic_league_code, confederation, total_clubs, url

=== countries (8 columnas) ===
country_id, country_name, country_code, confederation, total_clubs, total_players, average_age, url

=== game_events (11 columnas) ===

## 2.3. Carga de las tablas del alcance

De las 12 tablas disponibles se cargan las 9 que intervienen en el estudio. Se descartan
`game_events` (granularidad superior a la necesaria), `countries` y
`player_teammates_played_with` (sin aportación al objetivo).

In [6]:
TABLAS = ["appearances", "player_valuations", "players", "clubs",
          "competitions", "games", "game_lineups", "club_games"]

df = {}
for t in TABLAS:
    ruta = DATA / f"{t}.csv"
    if not ruta.exists():
        print(f"AVISO: no existe {t}.csv")
        continue
    df[t] = pd.read_csv(ruta, encoding="utf-8", low_memory=False)
    print(f"{t:<20} {df[t].shape[0]:>10,} x {df[t].shape[1]:>3}")

appearances           1,894,350 x  13
player_valuations       656,301 x   6
players                  50,149 x  26
clubs                       796 x  17
competitions                 65 x  11
games                    88,958 x  23
game_lineups          3,179,016 x  10
club_games              177,916 x  11


## 2.4. Decisión sobre cada columna

Antes de transformar nada se revisa columna a columna y se documenta una decisión justificada.

El catálogo completo se exporta al **Anexo A**.

In [7]:
# Decisiones por columna: usar / auxiliar / descartar, con su motivo.
DECISIONES = {
    "players": {
        "player_id":                    ("auxiliar",  "Clave primaria"),
        "name":                         ("auxiliar",  "Trazabilidad, no predictor"),
        "date_of_birth":                ("usar",      "Base para calcular la edad a fecha de corte"),
        "position":                     ("usar",      "Predictor y criterio de exclusión de porteros"),
        "sub_position":                 ("usar",      "Predictor: posición específica"),
        "foot":                         ("usar",      "Predictor: pie dominante"),
        "height_in_cm":                 ("usar",      "Predictor: perfil físico"),
        "country_of_citizenship":       ("usar",      "Predictor: nacionalidad"),
        "market_value_in_eur":          ("descartar", "Valor actual, contemporáneo al objetivo"),
        "highest_market_value_in_eur":  ("descartar", "Máximo histórico posterior a la fecha de corte"),
        "current_club_id":              ("descartar", "Club actual, no el de la temporada analizada"),
        "contract_expiration_date":     ("descartar", "Contrato vigente hoy, no el de la temporada"),
        "agent_name":                   ("descartar", "Alta cardinalidad y sin valor predictivo"),
        "image_url":                    ("descartar", "Sin valor predictivo"),
        "url":                          ("descartar", "Sin valor predictivo"),
    },
    "player_valuations": {
        "player_id":                    ("auxiliar",  "Clave foránea"),
        "date":                         ("usar",      "Define la ventana temporal del objetivo"),
        "market_value_in_eur":          ("usar",      "Variable objetivo"),
        "current_club_id":              ("auxiliar",  "Club en la fecha de la valoración"),
    },
    "appearances": {
        "appearance_id":                ("auxiliar",  "Clave primaria"),
        "game_id":                      ("auxiliar",  "Clave para unir con games"),
        "player_id":                    ("auxiliar",  "Clave de agrupación"),
        "player_club_id":               ("usar",      "Permite deducir el club de la temporada"),
        "competition_id":               ("usar",      "Tipo y nivel de competición"),
        "date":                         ("auxiliar",  "Coherencia temporal"),
        "minutes_played":               ("usar",      "Base de todas las métricas por 90 minutos"),
        "goals":                        ("usar",      "Producción ofensiva"),
        "assists":                      ("usar",      "Producción ofensiva"),
        "yellow_cards":                 ("usar",      "Disciplina"),
        "red_cards":                    ("usar",      "Disciplina"),
        "player_name":                  ("descartar", "Duplica un identificador ya presente"),
    },
    "clubs": {
        "club_id":                      ("auxiliar",  "Clave primaria"),
        "domestic_competition_id":      ("usar",      "Liga del club"),
        "stadium_seats":                ("usar",      "Proxy estable de dimensión del club"),
        "total_market_value":           ("descartar", "Valor actual de la plantilla, no el de la temporada"),
        "squad_size":                   ("descartar", "Tamaño actual de la plantilla"),
        "average_age":                  ("descartar", "Edad media actual de la plantilla"),
        "net_transfer_record":          ("descartar", "Saldo de fichajes actual"),
        "coach_name":                   ("descartar", "Entrenador actual, sin valor predictivo"),
    },
    "games": {
        "game_id":                      ("auxiliar",  "Clave primaria"),
        "season":                       ("usar",      "Asigna cada aparición a su temporada"),
        "competition_id":               ("auxiliar",  "Competición del partido"),
        "date":                         ("auxiliar",  "Coherencia temporal"),
        "home_club_id":                 ("auxiliar",  "Verificación de consistencia"),
        "away_club_id":                 ("auxiliar",  "Verificación de consistencia"),
    },
    "club_games": {
        "club_id":                      ("auxiliar",  "Clave"),
        "game_id":                      ("auxiliar",  "Clave"),
        "own_goals":                    ("usar",      "Rendimiento del club: goles a favor"),
        "opponent_goals":               ("usar",      "Rendimiento del club: goles en contra"),
        "is_win":                       ("usar",      "Rendimiento del club: puntos por partido"),
    },
    "game_lineups": {
        "game_id":                      ("auxiliar",  "Clave"),
        "player_id":                    ("auxiliar",  "Clave"),
        "type":                         ("usar",      "Titular o suplente"),
        "team_captain":                 ("usar",      "Rol de liderazgo"),
    },
    "competitions": {
        "competition_id":               ("auxiliar",  "Clave primaria"),
        "type":                         ("usar",      "Liga doméstica, copa o internacional"),
        "name":                         ("auxiliar",  "Legibilidad"),
        "country_name":                 ("auxiliar",  "Geografía"),
    },
}

filas = []
for tabla, cols in DECISIONES.items():
    if tabla not in df:
        continue
    presentes = set(df[tabla].columns)
    for col, (decision, motivo) in cols.items():
        filas.append({"tabla": tabla, "columna": col,
                      "existe": col in presentes,
                      "decision": decision, "motivo": motivo})
    # Columnas presentes en el fichero pero no contempladas
    for col in sorted(presentes - set(cols)):
        filas.append({"tabla": tabla, "columna": col, "existe": True,
                      "decision": "descartar", "motivo": "No contemplada en el alcance"})

catalogo = pd.DataFrame(filas)
catalogo.to_csv(PROCESSED / "anexoA_decisiones_columnas.csv",
                index=False, encoding="utf-8")

print("Resumen de decisiones:")
print(catalogo["decision"].value_counts().to_string())
print("\nColumnas declaradas que NO existen en el fichero (revisar):")
faltan = catalogo[~catalogo["existe"]]
print(faltan[["tabla", "columna"]].to_string(index=False) if len(faltan) else "  ninguna")

catalogo[catalogo["decision"] == "descartar"].head(20)

Resumen de decisiones:
decision
descartar    72
usar         24
auxiliar     21

Columnas declaradas que NO existen en el fichero (revisar):
  ninguna


,tabla,columna,existe,decision,motivo
8,players,market_value_in_eur,True,descartar,"Valor actual, contemporáneo al objetivo"
9,players,highest_market_value_in_eur,True,descartar,Máximo histórico posterior a la fecha de corte
10,players,current_club_id,True,descartar,"Club actual, no el de la temporada analizada"
11,players,contract_expiration_date,True,descartar,"Contrato vigente hoy, no el de la temporada"
12,players,agent_name,True,descartar,Alta cardinalidad y sin valor predictivo
13,players,image_url,True,descartar,Sin valor predictivo
14,players,url,True,descartar,Sin valor predictivo
15,players,city_of_birth,True,descartar,No contemplada en el alcance
16,players,country_of_birth,True,descartar,No contemplada en el alcance
17,players,current_club_domestic_competition_id,True,descartar,No contemplada en el alcance


---
# 3. Auditoría de calidad

La auditoría responde a una única pregunta: **¿soportan estos datos la transformación que se
pretende aplicar?**

Se examinan seis dimensiones: integridad de la adquisición, estructura y tipos, integridad
referencial, completitud, cobertura y consistencia interna.

## 3.1. Completitud

In [8]:
print("\n=== COLUMNAS CON AUSENCIAS RELEVANTES (> 5%) ===")
filas = []
for t in TABLAS:
    if t not in df:
        continue
    n = df[t].isna().mean()
    for col, pct in n[n > 0.05].items():
        filas.append({"tabla": t, "columna": col, "nulos_%": round(100*pct, 1)})
resumen = pd.DataFrame(filas).sort_values("nulos_%", ascending=False)
print(resumen.to_string(index=False) if len(resumen) else "  ninguna")

print("\n=== COLUMNAS CLAVE: ausencias esperadas cero ===")
CLAVES = [("appearances","player_id"), ("appearances","game_id"),
          ("appearances","minutes_played"), ("games","season"),
          ("games","game_id"), ("player_valuations","player_id"),
          ("player_valuations","market_value_in_eur"),
          ("club_games","club_id"), ("club_games","is_win"),
          ("clubs","domestic_competition_id")]
for t, c in CLAVES:
    if t in df and c in df[t].columns:
        k = df[t][c].isna().sum()
        print(f"  {t}.{c:<28} {k:>7,} nulos  {'OK' if k == 0 else '<-- REVISAR'}")


=== COLUMNAS CON AUSENCIAS RELEVANTES (> 5%) ===
            tabla                              columna  nulos_%
            clubs                   total_market_value   100.00
          players             current_national_team_id    93.50
          players                  international_goals    61.40
          players                   international_caps    61.40
            clubs                           coach_name    49.40
          players                           agent_name    46.50
          players             contract_expiration_date    37.00
            games                   away_club_position    28.50
            games                   home_club_position    28.50
       club_games                    opponent_position    28.50
       club_games                         own_position    28.50
     competitions                          total_clubs    23.10
     competitions                 domestic_league_code    18.50
     competitions                         country_name

## 3.2. Integridad referencial

Se verifica que las claves foráneas de cada tabla tienen correspondencia en su tabla de
referencia.

El criterio no es que el resultado sea cero, sino conocer la magnitud y haber decidido qué
hacer con ella.

In [9]:
def huerfanas(hija, col_h, madre, col_m):

    izq = set(df[hija][col_h].dropna().unique())
    der = set(df[madre][col_m].dropna().unique())
    faltan = izq - der
   
    filas = df[hija][col_h].isin(faltan).mean()
    return {"relación": f"{hija}.{col_h} → {madre}.{col_m}",
            "claves": len(izq), "huérfanas": len(faltan),
            "%_claves": round(100 * len(faltan) / max(len(izq), 1), 2),
            "%_filas":  round(100 * filas, 2)}

checks = [
    ("appearances",       "player_id",      "players",      "player_id"),
    ("appearances",       "game_id",        "games",        "game_id"),
    ("appearances",       "player_club_id", "clubs",        "club_id"),
    ("player_valuations", "player_id",      "players",      "player_id"),
    ("games",             "competition_id", "competitions", "competition_id"),
    ("games",             "home_club_id",   "clubs",        "club_id"),
    ("club_games",        "game_id",        "games",        "game_id"),
    ("club_games",        "club_id",        "clubs",        "club_id"),
    ("game_lineups",      "game_id",        "games",        "game_id"),
    ("game_lineups",      "player_id",      "players",      "player_id"),
]

integridad = pd.DataFrame([huerfanas(*c) for c in checks
                           if c[0] in df and c[2] in df])
integridad.to_csv(PROCESSED / "anexoB_integridad.csv", index=False, encoding="utf-8")

print("=== INTEGRIDAD REFERENCIAL ===")
print(integridad.to_string(index=False))

# RESULTADO
graves = integridad[integridad["%_filas"] > 5]
print(f"\nRelaciones con mas del 5% de FILAS afectadas: "
      f"{len(graves)} de {len(integridad)}")
if len(graves):
    print(graves[["relación", "%_claves", "%_filas"]].to_string(index=False))

CLAVES = [("appearances", "player_id"), ("appearances", "game_id"),
          ("appearances", "minutes_played"), ("appearances", "competition_id"),
          ("games", "game_id"), ("games", "season"),
          ("player_valuations", "player_id"),
          ("player_valuations", "market_value_in_eur"),
          ("club_games", "club_id"), ("club_games", "is_win"),
          ("clubs", "domestic_competition_id")]

print("\n=== COLUMNAS CRITICAS (ausencias esperadas: cero) ===")
for t, c in CLAVES:
    if t in df and c in df[t].columns:
        n = df[t][c].isna().sum()
        print(f"  {t}.{c:<28} {n:>7,} nulos   {'OK' if n == 0 else '<-- REVISAR'}")

print(f"\nCobertura temporal de games.season: "
      f"{df['games']['season'].min()} - {df['games']['season'].max()}")

=== INTEGRIDAD REFERENCIAL ===
                                          relación  claves  huérfanas  %_claves  %_filas
         appearances.player_id → players.player_id   29531          1      0.00     0.00
               appearances.game_id → games.game_id   73426          0      0.00     0.00
        appearances.player_club_id → clubs.club_id    1231        686     55.73     0.58
   player_valuations.player_id → players.player_id   41528          0      0.00     0.00
games.competition_id → competitions.competition_id      70          5      7.14     1.36
                games.home_club_id → clubs.club_id    2980       2187     73.39    14.41
                club_games.game_id → games.game_id   88958          0      0.00     0.00
                club_games.club_id → clubs.club_id    3274       2481     75.78    13.52
              game_lineups.game_id → games.game_id   81197          0      0.00     0.00
        game_lineups.player_id → players.player_id  114893      69943     60.88

## 3.3. Consistencia interna

Se comprueba que los registros no se contradicen entre sí ni consigo mismos. 

La ausencia  no relevante de incidencias es en sí misma un resultado, ya que garantiza que las
agregaciones posteriores no arrastran errores de origen.

In [10]:
incidencias = []

def chk(descripcion, n, total):
    incidencias.append({"comprobación": descripcion,
                        "casos": int(n),
                        "%": round(100 * n / max(total, 1), 3)})

# --- 1. Imposibles físicos ---------------------------------------------------
ap = df["appearances"]
n = len(ap)
chk("minutes_played negativo",            (ap["minutes_played"] < 0).sum(), n)
chk("minutes_played > 120",               (ap["minutes_played"] > 120).sum(), n)
chk("goals negativo",                     (ap["goals"] < 0).sum(), n)
chk("yellow_cards > 2 en un partido",     (ap["yellow_cards"] > 2).sum(), n)
chk("red_cards > 1 en un partido",        (ap["red_cards"] > 1).sum(), n)

pl = df["players"]
np_ = len(pl)
nac = pd.to_datetime(pl["date_of_birth"], errors="coerce")
chk("fecha de nacimiento nula o ilegible", nac.isna().sum(), np_)
chk("nacimiento fuera de 1950-2010",
    (~nac.dt.year.between(1950, 2010) & nac.notna()).sum(), np_)
chk("altura fuera de 150-215 cm",
    (~pl["height_in_cm"].between(150, 215) & pl["height_in_cm"].notna()).sum(), np_)

val = df["player_valuations"]
nv = len(val)
chk("valor de mercado <= 0", (val["market_value_in_eur"] <= 0).sum(), nv)

# --- 2. Coherencia temporal partido / temporada ------------------------------
g = df["games"].copy()
g["date"] = pd.to_datetime(g["date"])
ini = pd.to_datetime(g["season"].astype(str) + "-07-01")
fin = pd.to_datetime((g["season"] + 1).astype(str) + "-06-30")
chk("partido fuera del rango de su temporada",
    (~g["date"].between(ini, fin)).sum(), len(g))

# --- 3. Duplicados lógicos ---------------------------------------------------
chk("appearance_id duplicado", ap["appearance_id"].duplicated().sum(), n)
chk("mismo jugador dos veces en el mismo partido",
    ap.duplicated(["game_id", "player_id"]).sum(), n)
chk("misma valoración duplicada (jugador + fecha)",
    val.duplicated(["player_id", "date"]).sum(), nv)

# --- 4. Coherencia entre apariciones y partidos ------------------------------
cmp_ = ap[["game_id", "player_club_id"]].merge(
    df["games"][["game_id", "home_club_id", "away_club_id"]],
    on="game_id", how="inner")
incoherente = ~((cmp_["player_club_id"] == cmp_["home_club_id"]) |
                (cmp_["player_club_id"] == cmp_["away_club_id"]))
chk("club del jugador ajeno al partido", incoherente.sum(), len(cmp_))

# --- Resultado ---------------------------------------------------------------
auditoria = pd.DataFrame(incidencias)
auditoria.to_csv(PROCESSED / "anexoB_consistencia.csv", index=False, encoding="utf-8")

print("=== CONSISTENCIA INTERNA ===")
print(auditoria.to_string(index=False))

criticas = auditoria[auditoria["%"] > 3]
print("\nSin incidencias por encima del 3%: los datos son consistentes."
      if criticas.empty else f"\nATENCIÓN, revisar:\n{criticas.to_string(index=False)}")

=== CONSISTENCIA INTERNA ===
                                comprobación  casos    %
                     minutes_played negativo      0 0.00
                        minutes_played > 120      3 0.00
                              goals negativo      0 0.00
              yellow_cards > 2 en un partido      0 0.00
                 red_cards > 1 en un partido      0 0.00
         fecha de nacimiento nula o ilegible     49 0.10
               nacimiento fuera de 1950-2010      1 0.00
                  altura fuera de 150-215 cm     13 0.03
                       valor de mercado <= 0      1 0.00
     partido fuera del rango de su temporada   1940 2.18
                     appearance_id duplicado      0 0.00
 mismo jugador dos veces en el mismo partido      0 0.00
misma valoración duplicada (jugador + fecha)      0 0.00
           club del jugador ajeno al partido      0 0.00

Sin incidencias por encima del 3%: los datos son consistentes.


## 3.4. Cobertura temporal

Esta es la dimensión más determinante del estudio, ya que la variable objetivo se define
sobre una ventana temporal concreta. Se analiza cómo se distribuyen las valoraciones a lo
largo del tiempo y si la cobertura es homogénea durante todo el periodo.

In [11]:
pv = df["player_valuations"].copy()
COL_FECHA_VAL = next(c for c in ["date", "datetime"] if c in pv.columns)
pv[COL_FECHA_VAL] = pd.to_datetime(pv[COL_FECHA_VAL])

print("Rango temporal:", pv[COL_FECHA_VAL].min().date(), "→", pv[COL_FECHA_VAL].max().date())

print("\nValoraciones por año:")
print(pv[COL_FECHA_VAL].dt.year.value_counts().sort_index().loc[2010:].to_string())

print("\nDistribución por mes (identificación de oleadas de revisión):")
print(pv[COL_FECHA_VAL].dt.month.value_counts().sort_index().to_string())

print("\nValoraciones por jugador y año:")
print(pv.groupby(["player_id", pv[COL_FECHA_VAL].dt.year]).size()
        .describe()[["count", "mean", "50%", "max"]].to_string())

print("\nJugadores distintos CON VALORACIÓN, por año:")
print(pv.groupby(pv[COL_FECHA_VAL].dt.year)["player_id"].nunique().loc[2015:].to_string())

ap_ = df["appearances"].copy()
ap_["date"] = pd.to_datetime(ap_["date"])
print("\nJugadores distintos CON APARICIONES, por año:")
print(ap_.groupby(ap_["date"].dt.year)["player_id"].nunique().loc[2015:].to_string())

Rango temporal: 2000-01-20 → 2026-06-12

Valoraciones por año:
date
2010    13795
2011    17788
2012    19714
2013    22636
2014    23707
2015    30479
2016    34653
2017    35923
2018    41097
2019    46088
2020    48749
2021    58412
2022    59564
2023    57576
2024    45376
2025    45482
2026    20852

Distribución por mes (identificación de oleadas de revisión):
date
1      62993
2      41944
3      34511
4      27931
5      45898
6     149889
7      42406
8      27153
9      29497
10     40468
11     28508
12    125103

Valoraciones por jugador y año:
count   330,838.00
mean          1.98
50%           2.00
max          10.00

Jugadores distintos CON VALORACIÓN, por año:
date
2015    16473
2016    17794
2017    19155
2018    20162
2019    22116
2020    22807
2021    25650
2022    26848
2023    27890
2024    19628
2025    18951
2026    16711

Jugadores distintos CON APARICIONES, por año:
date
2015    7744
2016    7789
2017    7737
2018    7852
2019    7688
2020    7828
2021    7905

In [12]:
BIG5_DIAG = ["GB1", "ES1", "IT1", "L1", "FR1"]
apps_d = df["appearances"].merge(df["games"][["game_id", "season"]],
                                 on="game_id", how="left")
apps_d["date"] = pd.to_datetime(apps_d["date"])

print("\nCobertura de una ventana 1-may / 15-jul de 2024, por liga (temporada 2023/24):")
for lg in BIG5_DIAG:
    jug = set(apps_d.loc[(apps_d.season == 2023) &
                         (apps_d.competition_id == lg), "player_id"])
    v = pv[pv["player_id"].isin(jug) &
           pv[COL_FECHA_VAL].between("2024-06-01", "2024-07-15")]
    print(f"  {lg}: {len(jug):>4} jugadores | {v['player_id'].nunique():>4} con valoración "
          f"({v['player_id'].nunique()/max(len(jug),1):>6.1%})")

print("\nMes de revisión en 2024, por liga (jugadores distintos):")
for lg in BIG5_DIAG:
    jug = set(apps_d.loc[(apps_d.season == 2023) &
                         (apps_d.competition_id == lg), "player_id"])
    v24 = pv[pv["player_id"].isin(jug) & (pv[COL_FECHA_VAL].dt.year == 2024)]
    serie = v24.groupby(v24[COL_FECHA_VAL].dt.month)["player_id"].nunique()
    print(f"  {lg}: " + " | ".join(f"m{m}:{n}" for m, n in serie.items()))

print("\nÚltimo partido de la temporada 2023/24, por liga:")
for lg in BIG5_DIAG:
    s = apps_d[(apps_d.season == 2023) & (apps_d.competition_id == lg)]
    print(f"  {lg}: {s['date'].max().date()}")


Cobertura de una ventana 1-may / 15-jul de 2024, por liga (temporada 2023/24):
  GB1:  570 jugadores |   17 con valoración (  3.0%)
  ES1:  598 jugadores |  581 con valoración ( 97.2%)
  IT1:  590 jugadores |  570 con valoración ( 96.6%)
  L1:  494 jugadores |   28 con valoración (  5.7%)
  FR1:  525 jugadores |  491 con valoración ( 93.5%)

Mes de revisión en 2024, por liga (jugadores distintos):
  GB1: m3:155 | m5:518 | m6:17 | m7:9 | m9:23 | m10:107 | m11:1 | m12:488
  ES1: m2:1 | m3:233 | m5:9 | m6:581 | m7:7 | m9:10 | m10:117 | m12:462
  IT1: m3:147 | m5:18 | m6:570 | m7:7 | m9:4 | m10:101 | m11:2 | m12:473
  L1: m1:10 | m2:1 | m3:175 | m4:3 | m5:460 | m6:28 | m7:3 | m8:1 | m9:19 | m10:91 | m12:421
  FR1: m2:2 | m3:175 | m5:24 | m6:491 | m7:4 | m9:10 | m10:111 | m12:421

Último partido de la temporada 2023/24, por liga:
  GB1: 2024-05-19
  ES1: 2024-05-26
  IT1: 2024-06-02
  L1: 2024-05-18
  FR1: 2024-05-19


In [13]:
print("\nValoraciones en la ventana 1-may / 15-jul, por año:")
for anio in range(2016, 2026):
    v = pv[pv[COL_FECHA_VAL].between(f"{anio}-05-01", f"{anio}-07-15")]
    print(f"  {anio}: {len(v):>7,} valoraciones | "
          f"{v['player_id'].nunique():>6,} jugadores distintos")


Valoraciones en la ventana 1-may / 15-jul, por año:
  2016:   9,815 valoraciones |  9,607 jugadores distintos
  2017:  13,586 valoraciones | 13,543 jugadores distintos
  2018:  14,659 valoraciones | 14,568 jugadores distintos
  2019:  16,738 valoraciones | 16,693 jugadores distintos
  2020:   3,584 valoraciones |  3,580 jugadores distintos
  2021:  20,488 valoraciones | 20,361 jugadores distintos
  2022:  21,482 valoraciones | 21,458 jugadores distintos
  2023:  23,397 valoraciones | 23,322 jugadores distintos
  2024:  18,258 valoraciones | 18,245 jugadores distintos
  2025:  17,586 valoraciones | 17,582 jugadores distintos


## 3.5. Conclusiones de la auditoría

Los hallazgos y las decisiones que se derivan de ellos:

- **Oleadas de revisión**: las valoraciones no se actualizan de forma continua, sino en dos
oleadas anuales, una en primavera y otra en diciembre, coherente con una mediana de dos
valoraciones por jugador y año. La oleada de primavera no ocurre en una fecha única: se
distribuye a lo largo de múltiples fechas.

- **Las ligas no se revisan simultáneamente.** La Premier League (GB1) y la Bundesliga (L1) concluyen
antes que las ligas de España, Francia e Italia, y se revisan en **mayo**, mientras que LaLiga, la
Serie A y la Ligue 1 se revisan en **junio**. Para poder abarcar estas 5 grandes ligas, la ventana 
del objetivo se fija en **1 de mayo – 15 de julio**, tomando la última valoración disponible.

- **Cobertura estable desde 2016.** A partir de 2016 se superan sistemáticamente las 34.000
valoraciones anuales, mientras que los años anteriores son significativamente menores. En
consecuencia, el periodo de estudio será a partir de la temporada 2016/2017.

- **Anomalía de 2020.** La ventana de primavera de 2020 registra una fracción de las
valoraciones habituales, consecuencia de la interrupción de las competiciones por la
pandemia. Por ello, excluye la temporada 2019/2020.

- **Caída de cobertura desde 2024.** Los jugadores distintos con valoración descienden de
27.890 en 2023 a 19.628 en 2024, mientras que los jugadores con apariciones aumentan en
ese mismo periodo. En consecuencia, el conjunto de prueba se limita a la temporada 2023/2024 
y no se emplea 202420/25.

## 3.6. Checkpoint

Las tablas se guardan en formato Parquet, que conserva los tipos de datos y acelera
notablemente la lectura. A partir de aquí el proceso de construcción parte de estos ficheros,
lo que permite iterar sobre la transformación sin repetir la carga de los CSV.

In [14]:
for nombre, tabla in df.items():
    tabla.to_parquet(INTERIM / f"{nombre}.parquet", index=False)
    print(f"{nombre:<20} guardado")
print("\nCheckpoint completo en", INTERIM)

appearances          guardado
player_valuations    guardado
players              guardado
clubs                guardado
competitions         guardado
games                guardado
game_lineups         guardado
club_games           guardado

Checkpoint completo en e:\PEPO\PEPO\MASTER\TFM\tfm_estimacionValorMercadoFutbolista\entrega\implementacion\src\data\interim


---
# 4. Construcción del dataset analítico

## El diseño temporal

La decisión de ventana temporal para este proyecto es:

```
    Temporada S (2023 = 2023/24)          CORTE            VENTANA OBJETIVO
    [--- ago 2023  ...  may 2024 ---]   1 may 2024      [1 may - 15 jul 2024]
     <-------- PREDICTORES -------->                     <----- OBJETIVO ----->
```

- Ninguna variable explicativa puede usar infomración posterior al 1 de mayo.
- La variable objetivo representa la última valoración de un jugador en la temporada.
- La unidad de análisis es una fila que representa a un jugador en una temporada

## 4.1. Parámetros del estudio

Todos los parámetros están justificados en las conclusiones de la auditoría (sección 3.5) o
en el capítulo correspondiente de la memoria. Centralizarlos en una única celda permite
reproducir el estudio con otra configuración modificando un solo punto.

In [15]:
# --- Parámetros del estudio --------------------------------------------------
SEASONS          = list(range(2016, 2024))   # 2016/17 ... 2023/24
SEASON_EXCLUIR   = [2019]                    # Se excluye la temporada 2019/2020 por irregularidades a consecuencia del covid
BIG5             = ["GB1", "ES1", "IT1", "L1", "FR1"] # Se seleccionan para el estudio las 5 grandes ligas europeas
POSICIONES       = ["Attack", "Midfield", "Defender"] # Se descartan los porteros por las evidentes diferencias en rendimiento respecto al resto de la plantilla
MIN_MINUTOS      = 450                       # equivalente a 5 partidos completos
VENTANA_TARGET   = (5, 1, 7, 15)             # (mes_ini, dia_ini, mes_fin, dia_fin)
MES_CORTE, DIA_CORTE = 5, 1        # fin del periodo de rendimiento 
SPLIT            = {"train": [2016, 2017, 2018, 2020, 2021],
                    "valid": [2022],
                    "test":  [2023]}

SEASONS = [s for s in SEASONS if s not in SEASON_EXCLUIR]
print("Temporadas del estudio:", SEASONS)

# --- Carga desde los checkpoints ---------------------------------------------
tablas = ["appearances", "player_valuations", "players", "clubs",
          "competitions", "games", "game_lineups", "club_games"]

df = {}
for t in tablas:
    ruta = INTERIM / f"{t}.parquet"
    if ruta.exists():
        df[t] = pd.read_parquet(ruta)
        print(f"{t:<20} {df[t].shape[0]:>10,} x {df[t].shape[1]:>3}")
    else:
        print(f"AVISO: falta {t}.parquet")

# --- Detección defensiva de nombres de columna -------------------------------
def col_fecha_de(tabla, candidatas=("date", "datetime", "transfer_date")):
    for c in candidatas:
        if c in df[tabla].columns:
            return c
    raise KeyError(f"No encuentro columna de fecha en {tabla}")

COL_FECHA_VAL = col_fecha_de("player_valuations")
COL_FECHA_APP = col_fecha_de("appearances")
print(f"\nFecha en valuations: '{COL_FECHA_VAL}' | en appearances: '{COL_FECHA_APP}'")

df["player_valuations"][COL_FECHA_VAL] = pd.to_datetime(df["player_valuations"][COL_FECHA_VAL])
df["appearances"][COL_FECHA_APP] = pd.to_datetime(df["appearances"][COL_FECHA_APP])

Temporadas del estudio: [2016, 2017, 2018, 2020, 2021, 2022, 2023]
appearances           1,894,350 x  13
player_valuations       656,301 x   6
players                  50,149 x  26
clubs                       796 x  17
competitions                 65 x  11
games                    88,958 x  23
game_lineups          3,179,016 x  10
club_games              177,916 x  11

Fecha en valuations: 'date' | en appearances: 'date'


## 4.2. Agregación jugador-temporada

`appearances` no incluye la temporada, por lo que se obtiene cruzando con `games`. A
continuación se agrupa por jugador y temporada, sumando la producción de todos sus partidos.

Se conserva el desglose por tipo de competición, ya que la exposición a competición europea
es un indicador relevante del nivel del jugador.

In [16]:
# 'appearances' ya incluye competition_id: de 'games' solo se necesita 'season'.
apps = df["appearances"].merge(df["games"][["game_id", "season"]],
                               on="game_id", how="left")
assert "competition_id" in apps.columns

if "type" in df["competitions"].columns:
    apps = apps.merge(
        df["competitions"][["competition_id", "type"]].rename(columns={"type": "comp_type"}),
        on="competition_id", how="left")
else:
    apps["comp_type"] = np.nan

print("Tipos de competición encontrados:")
print(apps["comp_type"].value_counts(dropna=False).to_string())

apps = apps[apps["season"].isin(SEASONS)].copy()

# Corte de rendimiento: se excluyen los partidos posteriores al 1 de mayo de
# S+1, fecha de apertura de la ventana de valoracion. Sin este corte, las
# ultimas jornadas serian posteriores al target en las ligas que se revisan
# antes (Premier y Bundesliga).
apps["_corte"] = pd.to_datetime((apps["season"] + 1).astype(str) +
                                f"-{MES_CORTE:02d}-{DIA_CORTE:02d}")
antes = len(apps)
apps = apps[apps[COL_FECHA_APP] < apps["_corte"]].copy()
print(f"\nApariciones excluidas por posteriores al corte: "
      f"{antes - len(apps):,} ({(antes - len(apps)) / antes:.1%})")
print(f"Apariciones en las temporadas del estudio: {len(apps):,}")

apps["es_liga"] = (apps["comp_type"] == "domestic_league").astype(int)
apps["es_euro"] = apps["competition_id"].isin(["CL", "EL", "UCOL"]).astype(int)

perf = (apps.groupby(["player_id", "season"])
        .agg(partidos        = ("game_id", "nunique"),
             minutos         = ("minutes_played", "sum"),
             goles           = ("goals", "sum"),
             asistencias     = ("assists", "sum"),
             amarillas       = ("yellow_cards", "sum"),
             rojas           = ("red_cards", "sum"),
             partidos_liga   = ("es_liga", "sum"),
             partidos_euro   = ("es_euro", "sum"),
             n_competiciones = ("competition_id", "nunique"))
        .reset_index())

print(f"Filas jugador-temporada tras agregar: {len(perf):,}")
assert perf.duplicated(["player_id", "season"]).sum() == 0

Tipos de competición encontrados:
comp_type
domestic_league              1606102
domestic_cup                  118769
other                          91729
international_cup              60523
NaN                            14200
national_team_competition       3027

Apariciones excluidas por posteriores al corte: 73,340 (7.8%)
Apariciones en las temporadas del estudio: 872,010
Filas jugador-temporada tras agregar: 46,825


## 4.3. Club de la temporada, forma del club y rol

**No se emplea `players.current_club_id`**: refleja el club actual del jugador, no el que
tenía en la temporada analizada. Para un jugador que cambió de equipo —o que ya se ha
retirado— sería sencillamente incorrecto.

El club de cada temporada se deduce de **aquel en el que acumuló más minutos**, información
que procede directamente de las apariciones y por tanto es históricamente correcta.

In [17]:
# Calculo del club en el que jugo durante la temporada (mas minutos jugados)
club_temp = (apps.groupby(["player_id", "season", "player_club_id"])["minutes_played"]
             .sum().reset_index()
             .sort_values("minutes_played", ascending=False)
             .drop_duplicates(["player_id", "season"])
             .rename(columns={"player_club_id": "club_id"})
             [["player_id", "season", "club_id"]])

perf = perf.merge(club_temp, on=["player_id", "season"], how="left")
perf["liga"] = perf["club_id"].map(df["clubs"].set_index("club_id")["domestic_competition_id"])

# --- Forma del club en esa temporada -----------------------------------------
cg = df["club_games"].merge(df["games"][["game_id", "season", "competition_id", "date"]],
                            on="game_id", how="left")
cg = cg[cg["competition_id"].isin(BIG5)].copy()   # solo liga doméstica

# Mismo corte que en el rendimiento individual: la forma del club no puede
# incorporar jornadas posteriores a la fecha de corte.
cg["date"] = pd.to_datetime(cg["date"])
cg = cg[cg["date"] < pd.to_datetime((cg["season"] + 1).astype(str) +
                                    f"-{MES_CORTE:02d}-{DIA_CORTE:02d}")]

cg["empate"] = (cg["own_goals"] == cg["opponent_goals"]).astype(int)
cg["puntos"] = cg["is_win"] * 3 + cg["empate"]

forma = (cg.groupby(["club_id", "season", "competition_id"])
         .agg(cg_partidos=("puntos", "size"), cg_puntos=("puntos", "sum"),
              cg_gf=("own_goals", "sum"), cg_gc=("opponent_goals", "sum"))
         .reset_index())
forma = forma[forma["cg_partidos"] >= 10]         # se descartan aquellos equipos con menos de 10 partidos disputados
forma["club_ppp"]   = forma["cg_puntos"] / forma["cg_partidos"]
forma["club_dif_g"] = (forma["cg_gf"] - forma["cg_gc"]) / forma["cg_partidos"]
forma["club_rank"]  = forma.groupby(["competition_id", "season"])["cg_puntos"] \
                           .rank(ascending=False, method="min")

perf = perf.merge(forma[["club_id", "season", "club_ppp", "club_dif_g", "club_rank"]],
                  on=["club_id", "season"], how="left")

# Titularidad y capitanes
gl = df["game_lineups"].merge(df["games"][["game_id", "season"]], on="game_id", how="left")
gl = gl[gl["season"].isin(SEASONS)]
gl["es_titular"] = (gl["type"].astype(str).str.lower() == "starting_lineup").astype(int)
gl["es_capitan"] = gl["team_captain"].fillna(0).astype(int) \
                   if "team_captain" in gl.columns else 0

rol = (gl.groupby(["player_id", "season"])
       .agg(convocatorias=("game_id", "nunique"),
            titularidades=("es_titular", "sum"),
            capitanias   =("es_capitan", "sum"))
       .reset_index())

perf = perf.merge(rol, on=["player_id", "season"], how="left")
for c in ["convocatorias", "titularidades", "capitanias"]:
    perf[c] = perf[c].fillna(0)

print(f"Tras club, forma y rol: {len(perf):,} filas")
print(f"Sin liga asignada: {perf['liga'].isna().mean():.2%}")

Tras club, forma y rol: 46,825 filas
Sin liga asignada: 1.96%


## 4.4. Perfil del jugador y edad a fecha de corte

In [18]:
pl = df["players"].copy()
pl["date_of_birth"] = pd.to_datetime(pl["date_of_birth"], errors="coerce")

cols_perfil = [c for c in ["player_id", "date_of_birth", "position", "sub_position",
                           "foot", "height_in_cm", "country_of_citizenship"]
               if c in pl.columns]
perf = perf.merge(pl[cols_perfil], on="player_id", how="left")

perf["fecha_corte"] = pd.to_datetime((perf["season"] + 1).astype(str) +
                                     f"-{MES_CORTE:02d}-{DIA_CORTE:02d}")
perf["edad"] = (perf["fecha_corte"] - perf["date_of_birth"]).dt.days / 365.25

print(perf.groupby("season")["edad"]
      .describe()[["count", "mean", "min", "max"]].round(1).to_string())

          count  mean   min   max
season                           
2016   6,616.00 26.20 16.10 42.30
2017   6,567.00 26.30 16.40 42.50
2018   6,764.00 26.30 16.30 41.60
2020   6,971.00 26.20 16.10 43.60
2021   5,950.00 26.50 16.30 41.20
2022   6,936.00 26.20 15.80 42.10
2023   7,005.00 26.10 16.10 41.70


## 4.5. Valoración previa y valor de plantilla

Se recupera la última valoración conocida antes del inicio de la temporada.

Es un predictor legítimo: se sitúa unos diez meses antes de la fecha de corte. No obstante,
al capturar la persistencia del mercado, se reserva para la **especificación de robustez**
y no forma parte del modelo principal.

A partir de esta misma variable se deriva el valor medio de la plantilla del club en esa
temporada, que sustituye al valor actual de `clubs` como indicador de dimensión económica.

In [19]:
val = (df["player_valuations"][["player_id", COL_FECHA_VAL, "market_value_in_eur"]]
       .rename(columns={COL_FECHA_VAL: "fecha", "market_value_in_eur": "valor"})
       .dropna(subset=["valor"])
       .sort_values("fecha"))

perf["inicio_temporada"] = pd.to_datetime(perf["season"].astype(str) + "-08-01")

izq = perf[["player_id", "season", "inicio_temporada"]].sort_values("inicio_temporada")
mv_prev = pd.merge_asof(izq, val,
                        left_on="inicio_temporada", right_on="fecha",
                        by="player_id", direction="backward",
                        tolerance=pd.Timedelta(days=400)).rename(columns={"valor": "mv_prev"})

perf = perf.merge(mv_prev[["player_id", "season", "mv_prev", "fecha"]]
                    .rename(columns={"fecha": "mv_prev_fecha"}),
                  on=["player_id", "season"], how="left")



perf["dias_mv_prev"]    = (perf["inicio_temporada"] - perf["mv_prev_fecha"]).dt.days
perf["mv_prev_ausente"] = perf["mv_prev"].isna().astype(int)
print(perf["dias_mv_prev"].describe(percentiles=[.5, .9, .99]).round(0).to_string())
print(f"Emparejamientos de mas de un ano: {(perf['dias_mv_prev'] > 365).mean():.2%}")

# Valor medio de plantilla: histórico y coherente con la fecha de corte
plantilla = (perf.dropna(subset=["mv_prev"])
             .groupby(["club_id", "season"])["mv_prev"]
             .agg(club_valor_medio="mean", club_n_valorados="size").reset_index())
perf = perf.merge(plantilla, on=["club_id", "season"], how="left")

# Aforo: foto actual, pero el tamaño del estadio es muy estable en el tiempo.
# Se admite como proxy de dimensión del club y se declara en limitaciones.
if "stadium_seats" in df["clubs"].columns:
    perf["club_aforo"] = perf["club_id"].map(
        df["clubs"].set_index("club_id")["stadium_seats"])

print(f"Filas con valoración previa: {perf['mv_prev'].notna().mean():.1%}")

count   43,402.00
mean        65.00
std         52.00
min          0.00
50%         55.00
90%        115.00
99%        254.00
max        400.00
Emparejamientos de mas de un ano: 0.20%
Filas con valoración previa: 92.7%


## 4.6. Variable objetivo

Para cada temporada se selecciona la **última valoración** de cada jugador dentro de la
ventana comprendida entre el 1 de mayo y el 15 de julio del año siguiente. Es la valoración
vigente al abrir el mercado de fichajes de verano, una vez concluida la temporada.

La ventana abarca un periodo amplio porque, según se comprobó en la auditoría, la revisión de
mayo no se produce en una fecha concreta sino distribuida a lo largo del mes.

In [20]:
m_ini, d_ini, m_fin, d_fin = VENTANA_TARGET

filas_t = []
for s in SEASONS:
    ini = pd.Timestamp(year=s + 1, month=m_ini, day=d_ini)
    fin = pd.Timestamp(year=s + 1, month=m_fin, day=d_fin)
    sub = val[(val["fecha"] >= ini) & (val["fecha"] <= fin)].copy()
    sub = sub.sort_values("fecha").drop_duplicates("player_id", keep="last")
    sub["season"] = s
    filas_t.append(sub[["player_id", "season", "valor", "fecha"]])

target = (pd.concat(filas_t, ignore_index=True)
          .rename(columns={"valor": "target_valor", "fecha": "target_fecha"}))

print("Objetivos disponibles por temporada:")
print(target.groupby("season").size().to_string())

perf = perf.merge(target, on=["player_id", "season"], how="left")
print(f"\nFilas con objetivo: {perf['target_valor'].notna().sum():,} de {len(perf):,} "
      f"({perf['target_valor'].notna().mean():.1%})")

Objetivos disponibles por temporada:
season
2016    13543
2017    14568
2018    16693
2020    20361
2021    21458
2022    23322
2023    18245

Filas con objetivo: 42,842 de 46,825 (91.5%)


## 4.7. Ratios, filtros de población y deflactado

In [21]:
d = perf.copy()

# --- Métricas por 90 minutos -------------------------------------------------
noventas = d["minutos"] / 90.0
d["goles_90"]       = d["goles"] / noventas.replace(0, np.nan)
d["asistencias_90"] = d["asistencias"] / noventas.replace(0, np.nan)
d["ga_90"]          = (d["goles"] + d["asistencias"]) / noventas.replace(0, np.nan)
d["amarillas_90"]   = d["amarillas"] / noventas.replace(0, np.nan)

# --- Rol relativo ------------------------------------------------------------
d["min_por_partido"] = d["minutos"] / d["partidos"].replace(0, np.nan)
d["ratio_titular"]   = d["titularidades"] / d["convocatorias"].replace(0, np.nan)
d["jugo_europa"]     = (d["partidos_euro"] > 0).astype(int)

# --- Filtros de población ----------------------------------------------------
EMBUDO = [("Jugador-temporada inicial", len(d))]

d = d[d["target_valor"].notna()]         ; EMBUDO.append(("Con valoración objetivo", len(d)))
d = d[d["liga"].isin(BIG5)]              ; EMBUDO.append(("Ligas Big 5", len(d)))
d = d[d["minutos"] >= MIN_MINUTOS]       ; EMBUDO.append((f"Mínimo {MIN_MINUTOS} minutos", len(d)))
d = d[d["position"].isin(POSICIONES)]    ; EMBUDO.append(("Jugadores de campo", len(d)))
d = d[d["edad"].between(15, 45)]         ; EMBUDO.append(("Edad plausible", len(d)))

embudo = pd.DataFrame(EMBUDO, columns=["Criterio", "Filas"])
embudo["Retenido_%"] = (100 * embudo["Filas"] / embudo["Filas"].iloc[0]).round(1)
print("=== EMBUDO DE FILTRADO ===")
print(embudo.to_string(index=False))

=== EMBUDO DE FILTRADO ===
                 Criterio  Filas  Retenido_%
Jugador-temporada inicial  46825      100.00
  Con valoración objetivo  42842       91.50
              Ligas Big 5  17807       38.00
       Mínimo 450 minutos  13902       29.70
       Jugadores de campo  12866       27.50
           Edad plausible  12861       27.50


In [22]:
# # --- Indice de mercado (desfasado, sin informacion futura) -------------------
COL_COMP = "player_club_domestic_competition_id"
val_idx = (df["player_valuations"]
           .rename(columns={COL_FECHA_VAL: "fecha",
                            "market_value_in_eur": "valor"})
           .dropna(subset=["valor"]))
val_idx["fecha"] = pd.to_datetime(val_idx["fecha"])

if COL_COMP in val_idx.columns:
    val_idx = val_idx[val_idx[COL_COMP].isin(BIG5)]
    print(f"Indice restringido a Big 5: {len(val_idx):,} valoraciones")
else:
    print("AVISO: no existe la columna de competicion; indice sin restringir")

filas_idx = []
for s in SEASONS:
    ini = pd.Timestamp(year=s, month=m_ini, day=d_ini)
    fin = pd.Timestamp(year=s, month=m_fin, day=d_fin)
    v = val_idx[(val_idx["fecha"] >= ini) & (val_idx["fecha"] <= fin)]
    filas_idx.append({"season": s, "indice_mercado": v["valor"].median(),
                      "n_indice": len(v),
                      "ventana": f"{ini:%Y-%m-%d} / {fin:%Y-%m-%d}"})

indice = pd.DataFrame(filas_idx)

UMBRAL_N = 0.5 * indice["n_indice"].median()
malas = indice["n_indice"] < UMBRAL_N
if malas.any():
    print(f"Ventana escasa, indice interpolado: {indice.loc[malas,'season'].tolist()}")
    indice.loc[malas, "indice_mercado"] = np.nan
    indice["indice_mercado"] = indice["indice_mercado"].interpolate().bfill().ffill()

print("\n=== INDICE DE MERCADO (ventana del ano anterior, Big 5) ===")
print(indice.to_string(index=False))
assert indice["indice_mercado"].notna().all(), "Alguna temporada sin indice"

d = d.merge(indice[["season", "indice_mercado"]], on="season", how="left")


assert (d["target_valor"] > 0).all(), "Valores no positivos en el objetivo"

d["log_indice"]  = np.log(d["indice_mercado"])
d["target_defl"] = d["target_valor"] / d["indice_mercado"]

d["y_log"]      = np.log(d["target_valor"])
d["y_log_defl"] = d["y_log"] - d["log_indice"]

d["mv_prev_defl"] = d["mv_prev"] / d["indice_mercado"]
d["log_mv_prev"]  = np.where(d["mv_prev"].notna(),
                             np.log(d["mv_prev"]) - d["log_indice"], np.nan)

print(f"\nAsimetria nominal: {d['target_valor'].skew():.2f} | "
      f"log nominal: {d['y_log'].skew():.2f} | "
      f"log deflactado: {d['y_log_defl'].skew():.2f}")
print("\nDeriva residual (mediana del objetivo deflactado por temporada):")
print(d.groupby("season")["target_defl"].median().round(3).to_string())

Indice restringido a Big 5: 192,331 valoraciones
Ventana escasa, indice interpolado: [2020]

=== INDICE DE MERCADO (ventana del ano anterior, Big 5) ===
 season  indice_mercado  n_indice                 ventana
   2016      600,000.00      3499 2016-05-01 / 2016-07-15
   2017      750,000.00      4875 2017-05-01 / 2017-07-15
   2018      800,000.00      4993 2018-05-01 / 2018-07-15
   2020      800,000.00       890 2020-05-01 / 2020-07-15
   2021      800,000.00      6140 2021-05-01 / 2021-07-15
   2022      750,000.00      6293 2022-05-01 / 2022-07-15
   2023      700,000.00      6527 2023-05-01 / 2023-07-15

Asimetria nominal: 3.44 | log nominal: -0.03 | log deflactado: -0.03

Deriva residual (mediana del objetivo deflactado por temporada):
season
2016    5.83
2017    6.67
2018    8.75
2020    7.50
2021    7.50
2022    9.33
2023   10.00


## 4.8. Ensamblado final, partición temporal y guardado

Se seleccionan las columnas definitivas y se marca la partición.

La partición es temporal, no aleatoria, ya que un reparto aleatorio situaría temporadas futuras
en entrenamiento y pasadas en prueba, evaluando el modelo en condiciones que no se dan en la
práctica.

In [23]:
FEAT_NUM = [
    "edad", "height_in_cm",                                            # perfil
    "partidos", "minutos", "goles", "asistencias", "amarillas", "rojas",  # producción
    "goles_90", "asistencias_90", "ga_90", "amarillas_90",             # eficiencia
    "min_por_partido", "ratio_titular", "capitanias",                  # rol
    "partidos_liga", "partidos_euro", "n_competiciones", "jugo_europa",  # competición
    "club_ppp", "club_dif_g", "club_rank", "club_valor_medio", "club_aforo",  # club
]
FEAT_CAT = ["position", "sub_position", "foot", "liga"]
FEAT_MV  = ["log_mv_prev", "mv_prev_ausente"]

FEAT_NUM = [c for c in FEAT_NUM if c in d.columns]
FEAT_CAT = [c for c in FEAT_CAT if c in d.columns]

ID_COLS = ["player_id", "season", "club_id", "target_fecha",
           "indice_mercado", "dias_mv_prev"]
Y_COLS  = ["target_valor", "target_defl", "y_log", "y_log_defl"]

analitico = d[ID_COLS + FEAT_NUM + FEAT_CAT + FEAT_MV + Y_COLS].copy()

analitico["split"] = np.select(
    [analitico["season"].isin(SPLIT["train"]),
     analitico["season"].isin(SPLIT["valid"]),
     analitico["season"].isin(SPLIT["test"])],
    ["train", "valid", "test"], default="fuera")

analitico.to_parquet(PROCESSED / "dataset_analitico.parquet", index=False)
print("Guardado en", PROCESSED / "dataset_analitico.parquet")
print(f"Dimensión: {analitico.shape[0]:,} filas x {analitico.shape[1]} columnas")

Guardado en e:\PEPO\PEPO\MASTER\TFM\tfm_estimacionValorMercadoFutbolista\entrega\implementacion\src\data\processed\dataset_analitico.parquet
Dimensión: 12,861 filas x 41 columnas


In [24]:
print("=== FILAS POR TEMPORADA Y LIGA ===")
print(pd.crosstab(analitico["season"], analitico["liga"], margins=True).to_string())

print("\n=== EMBUDO DE FILTRADO ===")
print(embudo.to_string(index=False))

print("\n=== DIMENSIÓN Y PARTICIÓN ===")
print(f"Filas: {analitico.shape[0]:,} | Jugadores: {analitico['player_id'].nunique():,}")
print(analitico.groupby("split").size().reindex(["train","valid","test"]).to_string())

print("\n=== COBERTURA DE TARGET POR LIGA (temporada 2023) ===")
p23 = perf[perf.season == 2023]
print(p23.groupby("liga")["target_valor"].agg(
    n="size", con_target=lambda s: s.notna().sum(),
    pct=lambda s: f"{s.notna().mean():.1%}").to_string())

=== FILAS POR TEMPORADA Y LIGA ===
liga     ES1   FR1   GB1   IT1    L1    All
season                                     
2016     397   362   365   388   316   1828
2017     396   363   369   374   326   1828
2018     383   357   373   370   334   1817
2020     393   373   374   394   334   1868
2021     396   360   369   395   325   1845
2022     390   357   366   395   323   1831
2023     404   326   383   401   330   1844
All     2759  2498  2599  2717  2288  12861

=== EMBUDO DE FILTRADO ===
                 Criterio  Filas  Retenido_%
Jugador-temporada inicial  46825      100.00
  Con valoración objetivo  42842       91.50
              Ligas Big 5  17807       38.00
       Mínimo 450 minutos  13902       29.70
       Jugadores de campo  12866       27.50
           Edad plausible  12861       27.50

=== DIMENSIÓN Y PARTICIÓN ===
Filas: 12,861 | Jugadores: 4,543
split
train    9186
valid    1831
test     1844

=== COBERTURA DE TARGET POR LIGA (temporada 2023) ===
        n  con_

---
# 5. Verificación de la salida

Antes de dar por buena la transformación se comprueba que el conjunto generado cumple las
propiedades que se le presuponen. Estas verificaciones son la garantía de que los modelos del
notebook 2 se entrenan sobre datos correctos.

## 5.1. Comprobaciones de integridad

In [25]:
a = pd.read_parquet(PROCESSED / "dataset_analitico.parquet")

pruebas = []
def verifica(descripcion, condicion, detalle=""):
    pruebas.append({"comprobación": descripcion,
                    "resultado": "OK" if condicion else "FALLO",
                    "detalle": detalle})

verifica("La clave jugador-temporada es única",
         a.duplicated(["player_id", "season"]).sum() == 0,
         f"{a.duplicated(['player_id','season']).sum()} duplicados")
verifica("El objetivo no tiene valores ausentes",
         a["y_log_defl"].notna().all())
verifica("Ninguna temporada aparece en dos particiones",
         not (set(a.loc[a.split=="train","season"]) & set(a.loc[a.split=="test","season"])))
verifica("Todas las filas tienen partición asignada",
         (a["split"] != "fuera").all())
verifica("Edad dentro de un rango plausible",
         a["edad"].between(15, 45).all(),
         f"[{a['edad'].min():.1f}, {a['edad'].max():.1f}]")
verifica("Minutos por encima del umbral fijado",
         (a["minutos"] >= MIN_MINUTOS).all())
verifica("Ratio de titularidad entre 0 y 1",
         a["ratio_titular"].between(0, 1).all())
verifica("Sin ratios por 90 imposibles (goles_90 < 3)",
         (a["goles_90"] < 3).all(), f"máx {a['goles_90'].max():.2f}")
verifica("El objetivo es estrictamente positivo",
         (a["target_valor"] > 0).all())
verifica("Sin porteros en la muestra",
         "Goalkeeper" not in a["position"].unique())

resultado = pd.DataFrame(pruebas)
print(resultado.to_string(index=False))
print("\nTODAS LAS COMPROBACIONES SUPERADAS"
      if (resultado["resultado"] == "OK").all() else "\nHAY FALLOS: revisar")

print("\n--- Nulos por columna (solo las que tienen) ---")
nulos = a.isna().mean()
print((nulos[nulos > 0] * 100).round(2).to_string() if (nulos > 0).any() else "  ninguno")

                                comprobación resultado      detalle
         La clave jugador-temporada es única        OK 0 duplicados
       El objetivo no tiene valores ausentes        OK             
Ninguna temporada aparece en dos particiones        OK             
   Todas las filas tienen partición asignada        OK             
           Edad dentro de un rango plausible        OK [16.8, 43.6]
        Minutos por encima del umbral fijado        OK             
            Ratio de titularidad entre 0 y 1        OK             
 Sin ratios por 90 imposibles (goles_90 < 3)        OK     máx 1.73
       El objetivo es estrictamente positivo        OK             
                  Sin porteros en la muestra        OK             

TODAS LAS COMPROBACIONES SUPERADAS

--- Nulos por columna (solo las que tienen) ---
dias_mv_prev   1.17
height_in_cm   0.08
club_ppp       0.04
club_dif_g     0.04
club_rank      0.04
foot           0.12
log_mv_prev    1.17


## 5.2. Resumen del conjunto generado

Tablas formateadas para su incorporación directa a la memoria.

In [26]:
print("=== DIMENSIÓN ===")
print(f"Filas:               {a.shape[0]:,}")
print(f"Columnas:            {a.shape[1]}")
print(f"Jugadores distintos: {a['player_id'].nunique():,}")
print(f"Temporadas por jugador (media): {a.shape[0]/a['player_id'].nunique():.2f}")

print("\n=== EMBUDO DE FILTRADO (memoria, cap. 4) ===")
print(embudo.to_string(index=False))

print("\n=== REPARTO POR PARTICIÓN ===")
rep = (a.groupby("split").agg(filas=("player_id", "size"),
                             jugadores=("player_id", "nunique"),
                             temporadas=("season", "nunique"))
       .reindex(["train", "valid", "test"]))
print(rep.to_string())

print("\n=== FILAS POR TEMPORADA ===")
print(a.groupby(["season", "split"]).size().unstack(fill_value=0).to_string())

print("\n=== DISTRIBUCIÓN POR LIGA Y POSICIÓN ===")
print(pd.crosstab(a["liga"], a["position"], margins=True).to_string())

=== DIMENSIÓN ===
Filas:               12,861
Columnas:            41
Jugadores distintos: 4,543
Temporadas por jugador (media): 2.83

=== EMBUDO DE FILTRADO (memoria, cap. 4) ===
                 Criterio  Filas  Retenido_%
Jugador-temporada inicial  46825      100.00
  Con valoración objetivo  42842       91.50
              Ligas Big 5  17807       38.00
       Mínimo 450 minutos  13902       29.70
       Jugadores de campo  12866       27.50
           Edad plausible  12861       27.50

=== REPARTO POR PARTICIÓN ===
       filas  jugadores  temporadas
split                              
train   9186       3754           5
valid   1831       1831           1
test    1844       1844           1

=== FILAS POR TEMPORADA ===
split   test  train  valid
season                    
2016       0   1828      0
2017       0   1828      0
2018       0   1817      0
2020       0   1868      0
2021       0   1845      0
2022       0      0   1831
2023    1844      0      0

=== DISTRIBUCIÓN POR 

## 5.3. Trazabilidad: de los datos en crudo a la fila final

Se sigue el recorrido de un jugador concreto desde sus apariciones individuales hasta la fila
que lo representa en el conjunto analítico. La comprobación demuestra que la agregación es
correcta y permite validar manualmente el cálculo de las variables derivadas.

In [27]:
# Un jugador con muchos minutos en una temporada cualquiera
pid = a.query("season == 2022 and minutos > 2500")["player_id"].iloc[0]
nombre = df["players"].query("player_id == @pid")["name"].iloc[0]

print(f"=== {nombre} (player_id {pid}), temporada 2022/23 ===")

crudo = apps.query("player_id == @pid and season == 2022")
print(f"\n--- EN CRUDO: {len(crudo)} apariciones (se muestran 5) ---")
print(crudo[["date", "competition_id", "minutes_played", "goals", "assists"]]
      .head().to_string(index=False))

print(f"\nSumas manuales -> minutos: {crudo['minutes_played'].sum()}, "
      f"goles: {crudo['goals'].sum()}, asistencias: {crudo['assists'].sum()}")

fila = a.query("player_id == @pid and season == 2022")
print("\n--- FILA RESULTANTE EN EL CONJUNTO ANALÍTICO ---")
print(fila.T.to_string())

# Verificación aritmética de una variable derivada
g90 = fila["goles"].iloc[0] / (fila["minutos"].iloc[0] / 90)
print(f"\nComprobación: goles_90 calculado = {g90:.3f} | "
      f"almacenado = {fila['goles_90'].iloc[0]:.3f}")

=== Jesús Navas (player_id 15956), temporada 2022/23 ===

--- EN CRUDO: 41 apariciones (se muestran 5) ---
      date competition_id  minutes_played  goals  assists
2022-08-12            ES1              90      0        0
2022-08-27            ES1              65      0        0
2022-09-06             CL              90      0        0
2022-09-10            ES1              17      0        0
2022-09-18            ES1              90      0        0

Sumas manuales -> minutos: 2584, goles: 0, asistencias: 5

--- FILA RESULTANTE EN EL CONJUNTO ANALÍTICO ---
                                  279
player_id                       15956
season                           2022
club_id                           368
target_fecha      2023-06-13 00:00:00
indice_mercado             750,000.00
dias_mv_prev                    59.00
edad                            37.44
height_in_cm                   170.00
partidos                           41
minutos                          2584
goles             

---
## Cierre

El conjunto analítico está construido y verificado en
`data/processed/dataset_analitico.parquet`.

**Ficheros generados:**

| Fichero | Contenido |
|---|---|
| `dataset_analitico.parquet` | Conjunto de modelado, una fila por jugador y temporada |
| `anexoA_decisiones_columnas.csv` | Decisión y motivo para cada columna del origen |
| `anexoB_consistencia.csv` | Resultados de las comprobaciones de consistencia |

**Continúa en `02_modelos.ipynb`**, que parte de este conjunto para el análisis exploratorio
y el entrenamiento de los modelos.

In [28]:
print("=== PERFIL DEL CONJUNTO ANALÍTICO ===")
print(a[["edad","minutos","partidos","goles","asistencias","ga_90",
         "ratio_titular","club_ppp","club_rank","club_valor_medio",
         "log_mv_prev","target_valor","y_log_defl"]]
      .describe(percentiles=[.01,.25,.5,.75,.99]).T
      .round(2).to_string())

print("\n=== CATEGÓRICAS ===")
for c in ["position","sub_position","foot","liga"]:
    print(f"\n{c}: {a[c].nunique()} valores")
    print(a[c].value_counts(dropna=False).head(8).to_string())

print("\n=== COBERTURA DE log_mv_prev POR PARTICIÓN ===")
print(a.groupby("split")["log_mv_prev"].apply(lambda s: f"{s.notna().mean():.1%}").to_string())

=== PERFIL DEL CONJUNTO ANALÍTICO ===
                     count          mean           std        min         1%          25%          50%           75%           99%            max
edad             12,861.00         27.01          4.09      16.80      19.22        23.95        26.79         29.89         36.82          43.63
minutos          12,861.00      1,811.22        831.78     450.00     474.60     1,111.00     1,770.00      2,443.00      3,751.40       4,348.00
partidos         12,861.00         26.44          8.86       5.00       8.00        20.00        27.00         33.00         46.00          55.00
goles            12,861.00          2.86          4.33       0.00       0.00         0.00         1.00          4.00         20.00          49.00
asistencias      12,861.00          2.21          2.73       0.00       0.00         0.00         1.00          3.00         12.00          25.00
ga_90            12,861.00          0.24          0.24       0.00       0.00         0

In [29]:
# =============================================================
# Catálogo de variables del conjunto analítico
# Memoria: cap. 4.3  |  Anexo D
# =============================================================

CATALOGO = [
    # (variable, bloque, descripción, origen y cálculo)
    ("edad",             "Perfil", "Edad del jugador a la fecha de corte",
     "(fecha_corte − date_of_birth) / 365,25, con fecha_corte = 1 de mayo de S+1"),
    ("height_in_cm",     "Perfil", "Altura en centímetros", "players.height_in_cm"),
    ("position",         "Perfil", "Posición general (ataque, medio, defensa)", "players.position"),
    ("sub_position",     "Perfil", "Posición específica", "players.sub_position"),
    ("foot",             "Perfil", "Pie dominante", "players.foot; nulos como categoría propia"),

    ("partidos",         "Producción", "Partidos disputados en la temporada",
     "Recuento de game_id distintos en appearances"),
    ("minutos",          "Producción", "Minutos totales jugados",
     "Suma de minutes_played"),
    ("goles",            "Producción", "Goles marcados", "Suma de goals"),
    ("asistencias",      "Producción", "Asistencias", "Suma de assists"),
    ("amarillas",        "Producción", "Tarjetas amarillas", "Suma de yellow_cards"),
    ("rojas",            "Producción", "Expulsiones", "Suma de red_cards"),

    ("goles_90",         "Eficiencia", "Goles por cada 90 minutos jugados",
     "goles / (minutos / 90)"),
    ("asistencias_90",   "Eficiencia", "Asistencias por 90 minutos",
     "asistencias / (minutos / 90)"),
    ("ga_90",            "Eficiencia", "Contribución de gol por 90 minutos",
     "(goles + asistencias) / (minutos / 90)"),
    ("amarillas_90",     "Eficiencia", "Amarillas por 90 minutos",
     "amarillas / (minutos / 90)"),

    ("min_por_partido",  "Rol", "Minutos medios por partido disputado",
     "minutos / partidos"),
    ("ratio_titular",    "Rol", "Proporción de convocatorias como titular",
     "titularidades / convocatorias, desde game_lineups"),
    ("capitanias",       "Rol", "Partidos disputados como capitán",
     "Recuento de team_captain en game_lineups"),

    ("partidos_liga",    "Competición", "Partidos en competición doméstica",
     "Recuento de apariciones con type = domestic_league"),
    ("partidos_euro",    "Competición", "Partidos en competición europea",
     "Recuento de apariciones en CL, EL o UCOL"),
    ("n_competiciones",  "Competición", "Competiciones distintas disputadas",
     "Recuento de competition_id distintos"),
    ("jugo_europa",      "Competición", "Indicador de participación europea",
     "1 si partidos_euro > 0"),

    ("club_ppp",         "Club", "Puntos por partido del club en su liga",
     "(victorias×3 + empates) / partidos, desde club_games"),
    ("club_dif_g",       "Club", "Diferencia de goles por partido del club",
     "(goles a favor − en contra) / partidos, desde club_games"),
    ("club_rank",        "Club", "Posición del club en su liga",
     "Ranking por puntos dentro de competición y temporada"),
    ("club_valor_medio", "Club", "Valor medio de la plantilla del club",
     "Media de mv_prev de los jugadores del club en esa temporada"),
    ("club_aforo",       "Club", "Capacidad del estadio",
     "clubs.stadium_seats; foto actual, admitida por su estabilidad"),

    ("liga",             "Club", "Competición doméstica del jugador",
     "Liga en la que acumuló más minutos"),

    ("log_mv_prev",      "Historial", "Valoración previa, deflactada y en logaritmo",
     "ln(mv_prev) − ln(índice); solo en la especificación de robustez"),
]

catalogo = pd.DataFrame(CATALOGO,
                        columns=["variable", "bloque", "descripción", "origen y cálculo"])

# Se contrasta con las columnas realmente presentes en el conjunto generado
catalogo["en_dataset"] = catalogo["variable"].isin(a.columns)
faltan = set(FEAT_NUM + FEAT_CAT + FEAT_MV) - set(catalogo["variable"])

catalogo.to_csv(PROCESSED / "anexoC_catalogo_variables.csv",
                index=False, encoding="utf-8")

print(f"=== CATÁLOGO DE VARIABLES ({len(catalogo)} entradas) ===")
print(catalogo.groupby("bloque").size().to_string())
print(f"\nVariables del modelo sin documentar: {faltan if faltan else 'ninguna'}")
print(f"Entradas no presentes en el conjunto: "
      f"{catalogo.loc[~catalogo['en_dataset'], 'variable'].tolist() or 'ninguna'}")

catalogo.drop(columns="en_dataset")

=== CATÁLOGO DE VARIABLES (29 entradas) ===
bloque
Club           6
Competición    4
Eficiencia     4
Historial      1
Perfil         5
Producción     6
Rol            3

Variables del modelo sin documentar: {'mv_prev_ausente'}
Entradas no presentes en el conjunto: ninguna


,variable,bloque,descripción,origen y cálculo
0,edad,Perfil,Edad del jugador a la fecha de corte,"(fecha_corte − date_of_birth) / 365,25, con fe..."
1,height_in_cm,Perfil,Altura en centímetros,players.height_in_cm
2,position,Perfil,"Posición general (ataque, medio, defensa)",players.position
3,sub_position,Perfil,Posición específica,players.sub_position
4,foot,Perfil,Pie dominante,players.foot; nulos como categoría propia
5,partidos,Producción,Partidos disputados en la temporada,Recuento de game_id distintos en appearances
6,minutos,Producción,Minutos totales jugados,Suma de minutes_played
7,goles,Producción,Goles marcados,Suma de goals
8,asistencias,Producción,Asistencias,Suma de assists
9,amarillas,Producción,Tarjetas amarillas,Suma de yellow_cards
